# 📘 Cuaderno 01: Ingesta, Limpieza y Normalización Clínica Relacional
## Pipeline ETL de Fuentes Primarias para el Asistente Clínico Sofia
> **Fuentes Primarias Reales:**
> 1. **Dra. Gabriela Jufe:** *"Psicofarmacología Práctica"* (4ª Edición, 640 páginas).
> 2. **Dr. Stephen Stahl:** *"The Prescriber's Guide"* (Cambridge University Press).
> 3. **CANMAT / ISBD 2018 & CANMAT 2016:** Guías clínicas de práctica basada en la evidencia para Trastorno Bipolar y Depresión Mayor.
> 4. **AEMPS / Nomenclátor Oficial:** Catálogo de marcas comerciales y presentaciones de psicofármacos en España.
> 5. **Corpus Clínico Enriquecido:** 244 documentos estructurados (`pre_training/`).

---
### 🎯 Objetivos de este Cuaderno:
1. **Ingesta Híbrida y Reproducible:** Carga de documentos primarios con detección automática de PDF y fallback transparente a corpus curados.
2. **Estandarización Léxica y Semántica:** Limpieza de espacios, normalización Unicode y capitalización uniforme de variables textuales.
3. **Mapeo y Validación Sintáctica ATC OMS Nivel 5:** Resolución y aserción estricta de códigos de principio activo de 7 caracteres (`^[A-Z]\d{2}[A-Z]{2}\d{2}$`).
4. **Clasificación por Familias Farmacológicas Mayores:** Agrupación jerárquica limpia a partir del prefijo ATC de nivel 4 sin categorías solapadas.
5. **Cálculo de la Matriz Farmacocinética CYP450:** Extracción heurística y conteo dinámico de 15 flags enzimáticos booleanos + 1 ordinal de potencia.
6. **Normalización Relacional de Seguridad:** Desagregación de eventos adversos con vocabulario canónico (`Común / Moderado` y `Grave / Riesgo Vital`).
7. **Rigor Terminológico en Evidencia:** Tratamiento formal de recomendaciones CANMAT ("No Recomendado / Evidencia Negativa", 0 duplicados en claves compuestas).
8. **Persistencia Consolidada:** Exportación de datasets CSV limpios y sincronización transaccional con la base de datos relacional SQLite `sofia_clinica.db`.


## 1. Configuración de Entorno, Rutas Dinámicas y Vocabulario Canónico
Garantizamos la portabilidad de ejecución en cualquier entorno (directorio raíz o subcarpeta ) y establecemos el vocabulario clínico canónico.

In [3]:
%pip install requests pandas numpy matplotlib seaborn joblib scikit-learn

In [4]:
import os
import sys
import re
import json
import sqlite3
import unicodedata
from pathlib import Path
import pandas as pd
import numpy as np

# Detección dinámica y robusta de la raíz del proyecto
def _encontrar_raiz():
    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if (p / 'src').is_dir() and (p / 'data').is_dir():
            return p
        if (p / 'Sofia' / 'src').is_dir() and (p / 'Sofia' / 'data').is_dir():
            return p / 'Sofia'
    raise RuntimeError("No se pudo determinar la raíz del proyecto Sofia.")

_raiz = _encontrar_raiz()
if str(_raiz) not in sys.path:
    sys.path.insert(0, str(_raiz))

# Rutas del proyecto
PROJECT_ROOT = _raiz
DATA_DIR = PROJECT_ROOT / "data"
DATA_PROCESSED = DATA_DIR / "processed"
DATA_RAW = DATA_DIR / "raw"
DATA_CACHE = DATA_DIR / "cache"
PRE_TRAINING_DIR = PROJECT_ROOT / "pre_training"

# Vocabularios clínicos canónicos del proyecto Sofia
COMUN = "Común / Moderado"
GRAVE = "Grave / Riesgo Vital"
ORDEN_SEVERIDAD = [COMUN, GRAVE]
MAP_SEV = {COMUN: 1, GRAVE: 2}

ORDEN_EVIDENCIA_CANMAT = ["Level 4", "Level 3", "Level 2", "Level 1"]
DIRECCIONES_EVIDENCIA_VALIDAS = [
    "Recomendado / Primera Línea",
    "Recomendado / Segunda Línea",
    "Recomendado / Tercera Línea",
    "No Recomendado / Evidencia Negativa",
    "No Recomendado / Alto Riesgo de Viraje"
]

from src.extractors.jufe_extractor import JufeExtractor
from src.extractors.stahl_extractor import StahlExtractor
from src.extractors.canmat_extractor import CANMATExtractor
from src.enrichers.atc_enricher import ATCEnricher

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"✓ Entorno configurado correctamente.")
print(f"  • Raíz del proyecto: {PROJECT_ROOT.name}")
print(f"  • Vocabulario de severidad canónico: {ORDEN_SEVERIDAD}")
print(f"  • Niveles de evidencia CANMAT: {ORDEN_EVIDENCIA_CANMAT}")

✓ Entorno configurado correctamente.
  • Raíz del proyecto: Sofia
  • Vocabulario de severidad canónico: ['Común / Moderado', 'Grave / Riesgo Vital']
  • Niveles de evidencia CANMAT: ['Level 4', 'Level 3', 'Level 2', 'Level 1']


## 2. Ingesta, Limpieza y Normalización de Gabriela Jufe (*Psicofarmacología Práctica*)
La obra de la Dra. Jufe constituye la columna vertebral del catálogo psicofarmacológico. 
Implementamos un proceso de depuración en 3 pasos:
1. **Estandarización léxica:** Eliminación de dobles espacios y normalización de texto.
2. **Validación sintáctica ATC OMS Nivel 5:** Mediante la expresión regular oficial de 7 caracteres.
3. **Agrupación canónica por familias terapéuticas mayores:** 20 antidepresivos, 15 antipsicóticos, 8 antiepilépticos, 6 ansiolíticos, etc. (total: 57 fármacos sin categorías solapadas).


In [6]:
# 1. Ingesta estructurada (PDF de 640 páginas con fallback transparente)
jufe_extractor = JufeExtractor(raw_dir=DATA_RAW)
ruta_pdf = jufe_extractor.find_pdf_path()

if ruta_pdf and ruta_pdf.exists():
    print(f"✓ Archivo PDF localizado: {ruta_pdf.name} ({ruta_pdf.stat().st_size / (1024*1024):.2f} MB)")
    pdf_pages = jufe_extractor.pdf_extractor.load_or_extract_all_pages(ruta_pdf, max_pages=650)
    print(f"  • Páginas indexadas en memoria: {len(pdf_pages)}")
else:
    print("ℹ PDF no detectado en data/raw. Activando corpus de respaldo verificado de la obra de Jufe.")

df_jufe, df_jufe_efectos = jufe_extractor.extract_monographs()

# 2. Paso 1: Estandarización léxica
for col in ['farmaco', 'clase', 'mecanismo', 'posologia']:
    if col in df_jufe.columns:
        df_jufe[col] = df_jufe[col].astype(str).str.strip().apply(lambda s: re.sub(r'\s+', ' ', s))

# 3. Paso 2: Validación sintáctica de códigos ATC Nivel 5 de la OMS
df_jufe = ATCEnricher.enrich_dataframe(df_jufe, drug_column='farmaco')
atc_regex_n5 = re.compile(r'^[A-Z]\d{2}[A-Z]{2}\d{2}$')
validos_atc = df_jufe['atc_codigo'].apply(lambda x: bool(atc_regex_n5.match(str(x))))
assert validos_atc.all(), f"Códigos ATC inválidos detectados: {df_jufe.loc[~validos_atc, ['farmaco', 'atc_codigo']]}"

# 4. Paso 3: Agrupación canónica por familias terapéuticas mayores
mapa_familias_mayores = {
    'N06A': 'Antidepresivos (N06A)',
    'N05A': 'Antipsicóticos (N05A)',
    'N03A': 'Antiepilépticos / Estabilizadores (N03A)',
    'N05B': 'Ansiolíticos (N05B)',
    'N06D': 'Fármacos contra Demencia (N06D)',
    'N05C': 'Hipnóticos y Sedantes (N05C)',
    'N06B': 'Psicoestimulantes / TDAH (N06B)'
}

def asignar_familia_mayor(atc):
    if not isinstance(atc, str) or len(atc) < 4:
        return 'Otros Psicofármacos'
    pref = atc[:4]
    return mapa_familias_mayores.get(pref, 'Otros Psicofármacos')

df_jufe['familia_atc_mayor'] = df_jufe['atc_codigo'].apply(asignar_familia_mayor)

# 5. Normalización y verificación de severidad en eventos adversos
df_jufe_efectos['severidad_cat'] = pd.Categorical(
    df_jufe_efectos['severidad'], categories=ORDEN_SEVERIDAD, ordered=True
)
df_jufe_efectos['severidad_num'] = df_jufe_efectos['severidad'].map(MAP_SEV)
assert df_jufe_efectos['severidad_num'].notna().all(), "Error: Categorías de severidad no reconocidas"

print(f"\n✓ Extracción y normalización de Jufe completada:")
print(f"  • Principios activos catalogados: {len(df_jufe)}")
print(f"  • Validación ATC nivel 5 OMS: 100% conforme ({validos_atc.sum()}/{len(df_jufe)})")
print(f"  • Eventos adversos estructurados: {len(df_jufe_efectos)}")
conteo_sev = df_jufe_efectos['severidad'].value_counts()
print(f"  • Eventos de severidad '{GRAVE}': {(conteo_sev.get(GRAVE, 0)/len(df_jufe_efectos))*100:.1f}% ({conteo_sev.get(GRAVE, 0)}/{len(df_jufe_efectos)})")
print("\n=== DISTRIBUCIÓN POR FAMILIA ATC MAYOR (SIN SOLAPAMIENTOS) ===")
print(df_jufe['familia_atc_mayor'].value_counts())


✓ Archivo PDF localizado: Psicofarmacologia practica - G. Jufe.pdf (7.11 MB)
  • Páginas indexadas en memoria: 640
✓ PDF de Jufe cargado (640 páginas). Compilando base completa de 57 fármacos reales.

✓ Extracción y normalización de Jufe completada:
  • Principios activos catalogados: 57
  • Validación ATC nivel 5 OMS: 100% conforme (57/57)
  • Eventos adversos estructurados: 505
  • Eventos de severidad 'Grave / Riesgo Vital': 40.0% (202/505)

=== DISTRIBUCIÓN POR FAMILIA ATC MAYOR (SIN SOLAPAMIENTOS) ===
familia_atc_mayor
Antidepresivos (N06A)                       20
Antipsicóticos (N05A)                       15
Antiepilépticos / Estabilizadores (N03A)     8
Ansiolíticos (N05B)                          6
Fármacos contra Demencia (N06D)              4
Hipnóticos y Sedantes (N05C)                 3
Psicoestimulantes / TDAH (N06B)              1
Name: count, dtype: int64


## 3. Ingesta y Matriz Farmacocinética de Stephen Stahl (*Prescriber's Guide*)
Stahl aporta los perfiles farmacocinéticos esenciales sobre el complejo multienzimático del citocromo P450 (CYP450).
Implementamos el parser de monografías estructuradas y computamos dinámicamente:
- **15 flags booleanos CYP:** 5 inhibidores (`inhibe_cyp...`), 6 sustratos (`sustrato_cyp...`), 4 inductores (`inductor_cyp...`).
- **1 variable ordinal:** `potencia_inhibicion_2d6` (0 = Sin inhibición, 1 = Débil/Moderada, 2 = Potente).
- Normalización robusta de cadenas con `.str[:1].str.isupper()` y separación de listas con `' | '`.


In [8]:
# Extracción farmacocinética de Stahl
df_stahl, df_eventos_stahl = StahlExtractor.compilar_stahl()

# 1. Identificación y cálculo dinámico de variables enzimáticas CYP
cyp_inh_cols = sorted([c for c in df_stahl.columns if c.startswith('inhibe_cyp')])
cyp_sus_cols = sorted([c for c in df_stahl.columns if c.startswith('sustrato_cyp')])
cyp_ind_cols = sorted([c for c in df_stahl.columns if c.startswith('inductor_cyp')])
todas_flags_cyp = cyp_inh_cols + cyp_sus_cols + cyp_ind_cols

total_cyp_bool = len(todas_flags_cyp)
assert total_cyp_bool == 15, f"Discrepancia en flags CYP booleanos: {total_cyp_bool} != 15"
assert 'potencia_inhibicion_2d6' in df_stahl.columns, "Falta columna de potencia 2D6"

# 2. Resumen dinámico de la actividad enzimática
df_stahl['num_cyp_inhibidos'] = df_stahl[cyp_inh_cols].sum(axis=1)
df_stahl['num_cyp_sustrato'] = df_stahl[cyp_sus_cols].sum(axis=1)
df_stahl['num_cyp_inductores'] = df_stahl[cyp_ind_cols].sum(axis=1)
df_stahl['indice_complejidad_cyp'] = df_stahl[todas_flags_cyp].sum(axis=1)

# 3. Aserciones de seguridad farmacocinética centinela
cbz_row = df_stahl[df_stahl['farmaco'].str.contains('Carbamazepina', case=False)].iloc[0]
assert cbz_row['num_cyp_inductores'] >= 4, "Carbamazepina debe registrar inducción enzimática múltiple"

flx_row = df_stahl[df_stahl['farmaco'].str.contains('Fluoxetina', case=False)].iloc[0]
assert flx_row['inhibe_cyp2d6'] == 1 and flx_row['potencia_inhibicion_2d6'] == 'Potente', "Fluoxetina debe ser inhibidor potente de 2D6"

# 4. Normalización de eventos adversos de Stahl
df_eventos_stahl['severidad_cat'] = pd.Categorical(
    df_eventos_stahl['severidad'], categories=ORDEN_SEVERIDAD, ordered=True
)
df_eventos_stahl['severidad_num'] = df_eventos_stahl['severidad'].map(MAP_SEV)
assert df_eventos_stahl['severidad_num'].notna().all()

print(f"✓ Extracción de Stahl completada:")
print(f"  • Monografías farmacocinéticas procesadas: {len(df_stahl)}")
print(f"  • Variables booleanas CYP450: {len(cyp_inh_cols)} inh + {len(cyp_sus_cols)} sus + {len(cyp_ind_cols)} ind = {total_cyp_bool} flags")
print(f"  • Eventos adversos normalizados: {len(df_eventos_stahl)}")
conteo_sev_stahl = df_eventos_stahl['severidad'].value_counts()
print(f"  • Proporción de eventos graves: {(conteo_sev_stahl.get(GRAVE, 0)/len(df_eventos_stahl))*100:.1f}% ({conteo_sev_stahl.get(GRAVE, 0)}/{len(df_eventos_stahl)})")
print(f"\nTop 5 fármacos con mayor complejidad enzimática CYP:")
df_stahl[['farmaco', 'indice_complejidad_cyp', 'num_cyp_inhibidos', 'num_cyp_sustrato', 'num_cyp_inductores']].sort_values(
    by='indice_complejidad_cyp', ascending=False
).head(5)


✓ Extracción de Stahl completada:
  • Monografías farmacocinéticas procesadas: 22
  • Variables booleanas CYP450: 5 inh + 6 sus + 4 ind = 15 flags
  • Eventos adversos normalizados: 188
  • Proporción de eventos graves: 38.3% (72/188)

Top 5 fármacos con mayor complejidad enzimática CYP:


## 4. Ingesta y Directrices Basadas en Evidencia: CANMAT / ISBD 2018
Procesamos las directrices clínicas de CANMAT para Trastorno Bipolar y Depresión Mayor:
- **Diferenciación precisa de registros ATC:**
  1. *Fármacos específicos nivel 5 OMS:* ej. `N05AH04` (Quetiapina).
  2. *Combinaciones farmacológicas:* ej. `N05AN01 + N05AH04` (Litio + Quetiapina).
  3. *Recomendaciones de clase terapéutica:* ej. `N06AB` (ISRS monoterapia), `N06AA` (Tricíclicos), `N06A` (Antidepresivos en mantenimiento).
- **Auditoría de unicidad:** Evaluación por clave compuesta `(patologia, fase, linea, farmaco)`.
- **Rigor clínico y regulatorio:** Empleo de la dirección oficial *"No Recomendado / Evidencia Negativa"* o *"Alto Riesgo de Viraje"* (evitando confundirla con contraindicación absoluta) y distinción entre advertencias FDA Black Box y notas de seguridad AEMPS/EMA.


In [10]:
# Extracción de recomendaciones y alertas clínicas de CANMAT
canmat_extractor = CANMATExtractor(raw_dir=DATA_RAW)
df_canmat, df_alertas_canmat = canmat_extractor.extract_recommendations()

# 1. Auditoría de nulos en variables de decisión clínica
cols_criticas = ['patologia', 'fase', 'linea', 'estrategia', 'farmaco', 'nivel_evidencia', 'direccion_evidencia']
nulos_canmat = {c: int(df_canmat[c].isnull().sum()) for c in cols_criticas}
assert all(v == 0 for v in nulos_canmat.values()), f"Error: Nulos detectados en CANMAT: {nulos_canmat}"

# 2. Verificación de duplicados por clave compuesta
clave_compuesta = ['patologia', 'fase', 'linea', 'farmaco']
dups_compuestos = int(df_canmat.duplicated(subset=clave_compuesta).sum())
assert dups_compuestos == 0, f"Error: Directrices duplicadas en CANMAT: {dups_compuestos}"

# 3. Categorización y extracción de códigos ATC nivel 5
pat_n5 = re.compile(r'[A-Z]\d{2}[A-Z]{2}\d{2}')

def categorizar_registro_atc(val):
    if not isinstance(val, str) or not val.strip():
        return 'Sin código ATC'
    encontrados = pat_n5.findall(val)
    if len(encontrados) > 1:
        return 'Combinación de Fármacos'
    elif len(encontrados) == 1:
        return 'Fármaco Específico Nivel 5'
    else:
        return 'Clase Terapéutica General'

df_canmat['tipo_registro_atc'] = df_canmat['atc'].apply(categorizar_registro_atc)
df_canmat['codigos_nivel5'] = df_canmat['atc'].apply(lambda s: pat_n5.findall(str(s)))

codigos_unicos_canmat = sorted(set(pat_n5.findall(' '.join(df_canmat['atc'].dropna()))))

print(f"✓ Carga de directrices CANMAT completada:")
print(f"  • Recomendaciones estructuradas: {len(df_canmat)} filas")
print(f"  • Alertas clínicas de seguridad derivadas: {len(df_alertas_canmat)} filas")
print(f"  • Nulos en variables clínicas críticas: 0")
print(f"  • Duplicados por clave compuesta (patología + fase + línea + fármaco): 0")
print(f"  • Principios activos únicos nivel 5 OMS evaluados: {len(codigos_unicos_canmat)}")
print("\n=== DISTRIBUCIÓN DE TIPOS DE REGISTROS ATC EN CANMAT ===")
print(df_canmat['tipo_registro_atc'].value_counts())


✓ PDF de CANMAT cargado (74 páginas). Compilando 67 recomendaciones oficiales y 5 alertas.
✓ Carga de directrices CANMAT completada:
  • Recomendaciones estructuradas: 67 filas
  • Alertas clínicas de seguridad derivadas: 5 filas
  • Nulos en variables clínicas críticas: 0
  • Duplicados por clave compuesta (patología + fase + línea + fármaco): 0
  • Principios activos únicos nivel 5 OMS evaluados: 34

=== DISTRIBUCIÓN DE TIPOS DE REGISTROS ATC EN CANMAT ===
tipo_registro_atc
Fármaco Específico Nivel 5    54
Combinación de Fármacos       10
Clase Terapéutica General      3
Name: count, dtype: int64


## 5. Compilación del Corpus RAG y Catálogo de Marcas Oficiales (AEMPS)
Integramos los 244 documentos Markdown enriquecidos ubicados en `pre_training/`, incorporando metadatos YAML, resumen verbalizable (*voice snippets*) y catalogación de marcas comerciales del Nomenclátor español para trazabilidad prescriptiva.


In [12]:
# Compilación y consolidación de la Base de Conocimiento RAG (244 documentos clínicos)
from scripts.rebuild_knowledge_base import regenerar_knowledge_base

regenerar_knowledge_base()

kb_path = DATA_PROCESSED / 'rag_knowledge_base.json'
with open(kb_path, 'r', encoding='utf-8') as f:
    documentos_rag = json.load(f)

assert len(documentos_rag) == 244, f"Discrepancia en documentos del corpus: {len(documentos_rag)} != 244"

print(f"\n✓ Base de conocimiento RAG compilada y validada:")
print(f"  • Total documentos clínicos ricos: {len(documentos_rag)}")
print(f"  • Fuentes consolidadas: Jufe (57), Stahl (22), CANMAT (72), Marcas AEMPS (66), Pre-training (27)")
print(f"  • Guardado en: {kb_path.name}")


✓ PDF de Jufe cargado (640 páginas). Compilando base completa de 57 fármacos reales.
✓ PDF de CANMAT cargado (74 páginas). Compilando 67 recomendaciones oficiales y 5 alertas.
Compilando base de datos relacional SQLite en: /Users/anaisabecolinaarismendi/Documents/GitHub/Sofia/data/processed/sofia_clinica.db
✓ Base de datos SQLite sofia_clinica.db generada e indexada con éxito.
Documentos pre_training cargados: 27
Total de documentos clínicos consolidados en la Base de Conocimiento RAG: 244
✓ Guardado exitoso de 244 documentos en: /Users/anaisabecolinaarismendi/Documents/GitHub/Sofia/data/processed/rag_knowledge_base.json
✓ Verificación RAG: kb_obsoleta es False. Motor plenamente operativo y sincronizado.

✓ Base de conocimiento RAG compilada y validada:
  • Total documentos clínicos ricos: 244
  • Fuentes consolidadas: Jufe (57), Stahl (22), CANMAT (72), Marcas AEMPS (66), Pre-training (27)
  • Guardado en: rag_knowledge_base.json


## 6. Validación de Esquemas, Aserciones Centinela y Persistencia Dual (CSV + SQLite)
Exportamos las tablas normalizadas en formato CSV y las sincronizamos directamente en la base de datos relacional SQLite `sofia_clinica.db`, indexando las columnas de consulta clínica.


In [14]:
# 1. Preparación de DataFrames limpios para exportación
df_stahl_export = df_stahl.copy()
if 'efectos_comunes' in df_stahl_export.columns and isinstance(df_stahl_export['efectos_comunes'].iloc[0], list):
    df_stahl_export['efectos_comunes'] = df_stahl_export['efectos_comunes'].apply(lambda l: ' | '.join(l))
if 'efectos_graves' in df_stahl_export.columns and isinstance(df_stahl_export['efectos_graves'].iloc[0], list):
    df_stahl_export['efectos_graves'] = df_stahl_export['efectos_graves'].apply(lambda l: ' | '.join(l))

cols_canmat = ['patologia', 'fase', 'linea', 'estrategia', 'farmaco', 'atc', 'nivel_evidencia', 'direccion_evidencia', 'comentarios']

# 2. Persistencia en archivos CSV
df_jufe.to_csv(DATA_PROCESSED / 'jufe_farmacos_clean.csv', index=False, encoding='utf-8')
df_jufe_efectos.to_csv(DATA_PROCESSED / 'jufe_efectos_adversos_clean.csv', index=False, encoding='utf-8')
df_stahl_export.to_csv(DATA_PROCESSED / 'stahl_farmacos_clean.csv', index=False, encoding='utf-8')
df_eventos_stahl.to_csv(DATA_PROCESSED / 'stahl_eventos_adversos_clean.csv', index=False, encoding='utf-8')
df_canmat[cols_canmat].to_csv(DATA_PROCESSED / 'canmat_recomendaciones_clean.csv', index=False, encoding='utf-8')
df_alertas_canmat.to_csv(DATA_PROCESSED / 'canmat_alertas_clinicas_clean.csv', index=False, encoding='utf-8')
df_alertas_canmat.to_csv(DATA_PROCESSED / 'canmat_alertas_clean.csv', index=False, encoding='utf-8')

# 3. Sincronización e indexación en base de datos relacional SQLite (sofia_clinica.db)
con_db = sqlite3.connect(DATA_PROCESSED / 'sofia_clinica.db')
cur = con_db.cursor()

tablas_sql = {
    'jufe_farmacos': df_jufe,
    'jufe_efectos_adversos': df_jufe_efectos,
    'stahl_farmacos': df_stahl_export,
    'stahl_eventos_adversos': df_eventos_stahl,
    'canmat_recomendaciones': df_canmat[cols_canmat],
    'canmat_alertas': df_alertas_canmat
}

for nombre_tabla, df_t in tablas_sql.items():
    df_t.to_sql(nombre_tabla, con_db, if_exists='replace', index=False)
    if 'farmaco' in df_t.columns:
        cur.execute(f"CREATE INDEX IF NOT EXISTS idx_{nombre_tabla}_farmaco ON {nombre_tabla}(farmaco);")
    if 'atc_codigo' in df_t.columns:
        cur.execute(f"CREATE INDEX IF NOT EXISTS idx_{nombre_tabla}_atc ON {nombre_tabla}(atc_codigo);")

con_db.commit()
con_db.close()

print(f"✓ Pipeline ETL completado con éxito:")
print(f"  • jufe_farmacos_clean.csv: {df_jufe.shape}")
print(f"  • jufe_efectos_adversos_clean.csv: {df_jufe_efectos.shape}")
print(f"  • stahl_farmacos_clean.csv: {df_stahl_export.shape}")
print(f"  • stahl_eventos_adversos_clean.csv: {df_eventos_stahl.shape}")
print(f"  • canmat_recomendaciones_clean.csv: {df_canmat[cols_canmat].shape}")
print(f"  • canmat_alertas_clean.csv: {df_alertas_canmat.shape}")
print(f"  • Base de datos relacional SQLite 'sofia_clinica.db' sincronizada e indexada.")


✓ Pipeline ETL completado con éxito:
  • jufe_farmacos_clean.csv: (57, 14)
  • jufe_efectos_adversos_clean.csv: (505, 8)
  • stahl_farmacos_clean.csv: (22, 28)
  • stahl_eventos_adversos_clean.csv: (188, 7)
  • canmat_recomendaciones_clean.csv: (67, 9)
  • canmat_alertas_clean.csv: (5, 6)
  • Base de datos relacional SQLite 'sofia_clinica.db' sincronizada e indexada.
